In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain1-kp2000kd50-kpkdrand-26-norand'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'bp000-walking'
# exp_name = 'friction-walking-fractal-norand'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []
dof_pos_data = []
dof_vel_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {}}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()
dof_pos = env.dof_pos[0].cpu().numpy()
dof_vel = env.dof_vel[0].cpu().numpy()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())
dof_pos_data.append(dof_pos)
dof_vel_data.append(dof_vel)

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.0282, -0.1058,  0.2325, -0.0409, -0.7817, -0.0324,  0.0613, -0.0198,
          0.3866, -0.2790, -0.7475,  0.0842]], device='cuda:0')
Scaled actions :  tensor([[ 0.0282, -0.1058,  0.2325, -0.0409, -0.7817, -0.0324,  0.0613, -0.0198,
          0.3866, -0.2790, -0.7475,  0.0842]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7698e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06,  2.8184e-02, -1.0580e-01,
          2.3251e-01, -4.0947e-02, -7.8169e-01, -3.2383e-02,  6.1343e-02,
         -1.9835e-02,  3.8656e-01, -2.7905e-01, -7.4747e-01,  8.4234e-02]],
       device='cuda:0')
torques: [-1.58788519e-16 -1.00915103e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12612605e-17 -6.47408406e-18 -7.17891594e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.78307676e-17]
dof_pos: [-8.5621316e-08 -4.7636032e-08 -8.0014181e-01  1.6003647e+00
 -8.0019104e-01  7.6632642

In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.0111,  0.2397, -0.3327, -0.0527, -0.6923, -0.0283,  0.0652,  0.6697,
          0.1016, -0.1309, -0.9081,  0.0211]], device='cuda:0')
Scaled actions :  tensor([[-0.0111,  0.2397, -0.3327, -0.0527, -0.6923, -0.0283,  0.0652,  0.6697,
          0.1016, -0.1309, -0.9081,  0.0211]], device='cuda:0')
obs :  tensor([[ 0.2716, -0.6822, -0.1908, -0.0162, -0.0069, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0064, -0.0098,  0.0290,  0.0093, -0.0915, -0.0017,  0.0158,
         -0.0111,  0.0336,  0.0049, -0.0860,  0.0203,  0.0366, -0.0766,  0.2617,
          0.0477, -0.7379, -0.0203,  0.0908, -0.0772,  0.3029,  0.0026, -0.7129,
          0.0698, -0.0111,  0.2397, -0.3327, -0.0527, -0.6923, -0.0283,  0.0652,
          0.6697,  0.1016, -0.1309, -0.9081,  0.0211]], device='cuda:0')
torques: [-109.2353548   133.38565572  175.85697131 -200.           -5.72578021
 -104.51147474  -51.12575851  -45.02351555  -49.4431149  -200.
   18.01033495  -30.73612472]
dof_po

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.0356, -0.7230, -0.6679, -0.6657,  0.6360,  0.1058, -0.5142, -0.3701,
         -0.6840, -0.9030,  0.3470, -0.0466]], device='cuda:0')
Scaled actions :  tensor([[ 0.0356, -0.7230, -0.6679, -0.6657,  0.6360,  0.1058, -0.5142, -0.3701,
         -0.6840, -0.9030,  0.3470, -0.0466]], device='cuda:0')
obs :  tensor([[-2.8651e-01, -1.8899e-01, -1.4707e-01, -3.3454e-02, -5.7727e-03,
         -9.9942e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  5.1693e-03,
         -7.6879e-04,  6.7275e-02,  1.4980e-02, -2.8359e-01, -9.8648e-03,
          3.3552e-02, -3.7243e-03,  8.1435e-02,  9.1226e-03, -2.7807e-01,
          1.6341e-02, -2.4307e-02,  1.4571e-01,  1.2104e-01, -1.2446e-02,
         -9.0621e-01, -3.9811e-02,  7.4244e-02,  1.2271e-01,  1.6507e-01,
          4.0991e-02, -1.1319e+00, -3.1732e-02,  3.5640e-02, -7.2298e-01,
         -6.6790e-01, -6.6571e-01,  6.3602e-01,  1.0579e-01, -5.1425e-01,
         -3.7006e-01, -6.8400e-01, -9.0300e-01,  3.4701e-01, -4.6

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.0322, -0.0454,  1.2890,  0.1790,  0.5252,  0.0159,  0.0770,  1.0604,
          0.4855, -0.0336,  0.1417,  0.2932]], device='cuda:0')
Scaled actions :  tensor([[ 0.0322, -0.0454,  1.2890,  0.1790,  0.5252,  0.0159,  0.0770,  1.0604,
          0.4855, -0.0336,  0.1417,  0.2932]], device='cuda:0')
obs :  tensor([[ 0.3205,  0.4429,  0.1399, -0.0267, -0.0076, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0134, -0.0046,  0.0656,  0.0155, -0.3478,  0.0385, -0.0093,
         -0.0078,  0.0912,  0.0033, -0.3942, -0.0141,  0.0628, -0.1410, -0.1230,
          0.0249,  0.1357,  0.4086, -0.4410, -0.1408, -0.0618, -0.0297, -0.1146,
         -0.1744,  0.0322, -0.0454,  1.2890,  0.1790,  0.5252,  0.0159,  0.0770,
          1.0604,  0.4855, -0.0336,  0.1417,  0.2932]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          200.
  -77.94544897   49.3398761  -200.         -200.         -200.
  200.          164.58142989]
dof_pos: [ 0.0

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    dof_pos = env.dof_pos[0].cpu().numpy()
    dof_vel = env.dof_vel[0].cpu().numpy()
    print("torques:", torques)
    print("dof_pos:", dof_pos)
    print("dof_vel:", dof_vel)

    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    dof_pos_data.append(dof_pos)
    dof_vel_data.append(dof_vel)

    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.3511,  0.0732, -0.0207, -0.2384, -0.2620, -0.2517,  1.0652,  0.1579,
          0.2742, -0.4380,  0.3643, -0.2353]], device='cuda:0')
Scaled actions :  tensor([[-0.3511,  0.0732, -0.0207, -0.2384, -0.2620, -0.2517,  1.0652,  0.1579,
          0.2742, -0.4380,  0.3643, -0.2353]], device='cuda:0')
obs :  tensor([[ 0.0369, -0.2272, -0.1153, -0.0237, -0.0144, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0241, -0.0211,  0.0610,  0.0326, -0.2158,  0.0454, -0.0396,
         -0.0224,  0.1092, -0.0112, -0.3124,  0.0470,  0.0413, -0.0340,  0.0594,
          0.1311,  1.0890, -0.0630,  0.0933, -0.0130,  0.2100, -0.0903,  0.8376,
          0.5204, -0.3511,  0.0732, -0.0207, -0.2384, -0.2620, -0.2517,  1.0652,
          0.1579,  0.2742, -0.4380,  0.3643, -0.2353]], device='cuda:0')
torques: [196.99043421 200.         200.          39.03534584 198.53335522
 -14.41998185 -24.91240498  -7.22043657 200.         174.70652849
 200.           2.2057964 ]
dof_pos: [

In [26]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        dof_pos = env.dof_pos[0].cpu().numpy()
        dof_vel = env.dof_vel[0].cpu().numpy()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        dof_pos_data.append(dof_pos)
        dof_vel_data.append(dof_vel)
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.882, Scaled action max=0.882
Step 1/100, Total steps: 806
steps: 806
actions : tensor([[ 0.3677, -0.0163,  0.1127,  0.3177,  0.2431,  0.2778, -0.3781,  0.4502,
         -0.1850,  0.8364,  0.8819, -0.3624]], device='cuda:0')
target_dof_pos: tensor([[ 0.1147, -0.4715, -1.5734,  1.0293, -1.0100,  0.7512,  0.6440, -0.0929,
         -0.9344,  1.2299, -0.6524, -0.2779]], device='cuda:0')
Step 1: Original action max=0.985, Scaled action max=0.985
Step 2: Original action max=0.607, Scaled action max=0.607
Step 21/100, Total steps: 826
steps: 826
actions : tensor([[ 0.6993, -0.9367, -0.0997, -0.6187, -0.5519,  0.0289, -0.7401, -0.6327,
          1.9153,  0.3759,  0.9218, -0.3395]], device='cuda:0')
target_dof_pos: tensor([[ 0.3257,  0.2775, -0.7212,  1.2285, -2.3688,  0.5724,  0.6275,  0.8177,
          0.9237,  2.3648, -0.0278, -0.2312]], device='cuda:0')
Step 41/100, Total steps: 846
steps: 846
actions : tensor([[ 0.4236,  0.0862,  1.0888, -0.3908,  0.5898, -0.15

In [27]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [28]:
env.sim.stop()

In [31]:
env.reset()
cnt = 0

In [20]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]

    # dof_posデータ
    dof_pos_array = np.array(dof_pos_data)
    for i in range(dof_pos_array.shape[1]):
        data_dict[f'dof_pos_{i}'] = dof_pos_array[:, i]

    # dof_velデータ
    dof_vel_array = np.array(dof_vel_data)
    for i in range(dof_vel_array.shape[1]):
        data_dict[f'dof_vel_{i}'] = dof_vel_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-8_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (501, 82)
